In [1]:
from PIL import Image, ImageDraw, ImageFont
import random

In [2]:
def fill_form_by_mask(
    background_path,
    mask_path,
    font_path,
    output_path,
    x0, y0, dx, dy, w, h,
    alphabet="АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ0123456789",
    threshold=128,
    jitter=2,
    font_size=None,
    text_color=(0, 0, 0, 255),
):
    img = Image.open(background_path).convert("RGBA")
    draw = ImageDraw.Draw(img)

    mask = Image.open(mask_path).convert("L")

    mask = mask.resize((w, h), resample=Image.NEAREST)
    mask = mask.point(lambda p: 255 if p < threshold else 0)
    mask_px = mask.load()

    if font_size is None:
        font_size = int(dy * 0.85)
    font = ImageFont.truetype(font_path, size=font_size)

    for row in range(h):
        for col in range(w):
            if mask_px[col, row] == 0:
                continue

            # координаты клетки на бланке
            cell_x = x0 + col * dx
            cell_y = y0 + row * dy

            char = random.choice(alphabet)

            # центрирование символа в клетке
            bbox = draw.textbbox((0, 0), char, font=font)
            tw = bbox[2] - bbox[0]
            th = bbox[3] - bbox[1]
            # bbox[0], bbox[1] — смещение глифа относительно точки рисования
            pos_x = cell_x + (dx - tw) / 2 - bbox[0] + random.randint(-jitter, jitter)
            pos_y = cell_y + (dy - th) / 2 - bbox[1] + random.randint(-jitter, jitter)

            draw.text((pos_x, pos_y), char, font=font, fill=text_color)

    img.convert("RGB").save(output_path)

In [3]:
fill_form_by_mask(
    background_path="blank_2.jpg",
    mask_path="mask.jpg",
    font_path="Caveat-VariableFont_wght.ttf",
    output_path="output.jpg",
    x0=46, y0=377,
    dx=29.3, dy=29.3,
    w=39, h=44,
    threshold=128,
    jitter=1,
    font_size=26,
)